# Pinecone RAG — Section-Aware Clinical Trial Retrieval Pipeline

**Architecture:**
- Namespace structure: `{drug}_chunks` = full paragraph chunks, `{drug}_sections` = section summaries
- Retrieval is **section-by-section**, matching the document generation loop
- For each target section, we query the `_sections` namespace first (structural match),
  then the `_chunks` namespace (detailed content match), and merge both
- Namespaces are selected automatically from user metadata — no hardcoding needed
- Output: one structured LLM prompt per section, ready to pass to Groq


## 0. Install dependencies

In [ ]:
!pip install pinecone-client sentence-transformers pandas tabulate --quiet

## 1. Config

In [2]:
PINECONE_API_KEY = "pcsk_5K2TpT_AMzJu3syYmv3k9W5HkqatuyaaTcScNyd9sxvzkM4WUsW8JjB9NYZ8TSopuSeDZN"
PINECONE_INDEX   = "clinical-rag-index"
EMBED_MODEL      = "BAAI/bge-base-en-v1.5"
VECTOR_DIM       = 768

# All known namespaces — format: {drug}_chunks (full text) and {drug}_sections (summaries)
ALL_NAMESPACES = [
    'Chenodeoxycholic_chunks',
    'Chenodeoxycholic_sections',
    'Bimervax_chunks',
    'Bimervax_sections',
    'Amlodipine_chunks',
    'Amlodipine_sections',
    'Epclusa_v1_chunks',
    'Epclusa_v1_sections',
]

# Derived: pairs of (sections_ns, chunks_ns) per document
# sections_ns → section-level summaries (used for namespace scoring + structural retrieval)
# chunks_ns   → paragraph-level detail   (used for content retrieval)
NS_PAIRS = {
    'Chenodeoxycholic': ('Chenodeoxycholic_sections', 'Chenodeoxycholic_chunks'),
    'Bimervax':         ('Bimervax_sections',         'Bimervax_chunks'),
    'Amlodipine':       ('Amlodipine_sections',       'Amlodipine_chunks'),
    'Epclusa_v1':       ('Epclusa_v1_sections',       'Epclusa_v1_chunks'),
}

print(f"Config loaded — index: {PINECONE_INDEX}, model: {EMBED_MODEL}")
print(f"Documents: {list(NS_PAIRS.keys())}")


Config loaded — index: clinical-rag-index, model: BAAI/bge-base-en-v1.5
Documents: ['Chenodeoxycholic', 'Bimervax', 'Amlodipine', 'Epclusa_v1']


## 2. Connect to Pinecone

In [3]:
from pinecone import Pinecone

pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)

stats = index.describe_index_stats()
print(f"Connected  total_vectors={stats['total_vector_count']:,}  dim={stats['dimension']}")
print(f"Namespaces: {list(stats.get('namespaces', {}).keys())}")

assert stats['dimension'] == VECTOR_DIM, (
    f"Dimension mismatch: index={stats['dimension']} expected={VECTOR_DIM}"
)


c:\Users\BasanthKumarPutta\Downloads\draft\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected  total_vectors=1,091  dim=768
Namespaces: ['Bimervax_chunks', 'Chenodeoxycholic_chunks', 'Amlodipine_sections', 'Amlodipine_chunks', 'Chenodeoxycholic_sections', 'Epclusa_v1_sections', 'Bimervax_sections', 'Epclusa_v1_chunks']


## 3. Load encoder

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer(EMBED_MODEL)
print(f"Encoder ready: {EMBED_MODEL}")

def embed(text: str) -> list:
    """Embed one string with L2 normalisation (matches FAISS IndexFlatIP)."""
    return encoder.encode([text], normalize_embeddings=True)[0].tolist()


Encoder ready: BAAI/bge-base-en-v1.5


## 4. SECTION_MAP — section-to-keyword mapping

Each entry defines:
- `title`    : human-readable section title used in the LLM prompt
- `keywords` : domain terms that bias the embedding toward content relevant for that section

These keywords are **prepended to the metadata query** before embedding so the
vector lands in the semantic neighbourhood of that section type, even when the
stored chunks have no `section_type` metadata tag.


In [5]:
SECTION_MAP = {
    "intro": {
        "title":    "1. Introduction and Rationale",
        "keywords": "introduction background rationale disease overview pathophysiology unmet need",
    },
    "study_design": {
        "title":    "2. Study Design",
        "keywords": "study design randomized controlled double-blind treatment arms blinding multicenter",
    },
    "objectives": {
        "title":    "3. Study Objectives and Endpoints",
        "keywords": "primary endpoint secondary endpoint objectives efficacy outcome measure",
    },
    "population": {
        "title":    "4. Study Population",
        "keywords": "inclusion exclusion criteria patient population eligibility diagnosis age",
    },
    "treatment": {
        "title":    "5. Treatment Plan",
        "keywords": "dosage dose regimen administration route treatment duration investigational product",
    },
    "safety": {
        "title":    "6. Safety and Monitoring",
        "keywords": "adverse events safety monitoring SAE DSMB stopping rules tolerability",
    },
    "statistics": {
        "title":    "7. Statistical Methodology",
        "keywords": "statistical analysis sample size power calculation ITT per-protocol",
    },
    "ethics": {
        "title":    "8. Ethical Considerations",
        "keywords": "informed consent IRB ethics committee regulatory GCP ICH E6",
    },
    "data_management": {
        "title":    "9. Data Management",
        "keywords": "data collection CRF EDC database quality control monitoring",
    },
}

print(f"{len(SECTION_MAP)} sections defined: {list(SECTION_MAP.keys())}")


9 sections defined: ['intro', 'study_design', 'objectives', 'population', 'treatment', 'safety', 'statistics', 'ethics', 'data_management']


## 5. Sample study metadata

In [6]:
SAMPLE_METADATA = {
    "indication":              "Cerebrotendinous Xanthomatosis (CTX)",
    "therapeutic_area":        "Rare Metabolic Disorders / Neurology",
    "phase":                   "Phase III",
    "design":                  "Randomized, Double-blind, Placebo-controlled, Multicenter",
    "primary_endpoint":        "Reduction in plasma cholestanol levels",
    "secondary_endpoints": [
        "Neurological function improvement (disability scales)",
        "MRI-based disease progression",
        "Bone mineral density changes",
        "Safety and tolerability (AE/SAE incidence)",
    ],
    "patient_population":      "Patients aged 2-75 years diagnosed with CTX",
    "investigational_product": "Chenodeoxycholic Acid (CDCA)",
    "sample_size":             120,
    "duration_months":         36,
    "treatment_arms":          ["CDCA 750 mg/day orally", "Placebo"],
    "sponsor":                 "Leadiant Biosciences",
    "protocol_number":         "CTX-CDCA-2024-001",
}

def build_metadata_query(metadata: dict) -> str:
    """
    Convert study metadata to a single query string.
    Mirrors StudyMetadata.to_query_text() in backend/core/models.py.
    """
    parts = [
        f"Indication: {metadata.get('indication', '')}",
        f"Phase: {metadata.get('phase', '')}",
        f"Design: {metadata.get('design', '')}",
        f"Primary Endpoint: {metadata.get('primary_endpoint', '')}",
        f"Population: {metadata.get('patient_population', '')}",
    ]
    if metadata.get('secondary_endpoints'):
        eps = metadata['secondary_endpoints']
        if isinstance(eps, list):
            eps = '; '.join(eps)
        parts.append(f"Secondary Endpoints: {eps}")
    if metadata.get('investigational_product'):
        parts.append(f"Drug: {metadata['investigational_product']}")
    if metadata.get('therapeutic_area'):
        parts.append(f"Therapeutic Area: {metadata['therapeutic_area']}")
    return " | ".join(parts)

meta_query = build_metadata_query(SAMPLE_METADATA)
print("Metadata query:")
print(meta_query)


Metadata query:
Indication: Cerebrotendinous Xanthomatosis (CTX) | Phase: Phase III | Design: Randomized, Double-blind, Placebo-controlled, Multicenter | Primary Endpoint: Reduction in plasma cholestanol levels | Population: Patients aged 2-75 years diagnosed with CTX | Secondary Endpoints: Neurological function improvement (disability scales); MRI-based disease progression; Bone mineral density changes; Safety and tolerability (AE/SAE incidence) | Drug: Chenodeoxycholic Acid (CDCA) | Therapeutic Area: Rare Metabolic Disorders / Neurology


## 6. Step 1 — Automatic namespace / document selection

**Problem:** The user fills in study metadata (indication, drug, phase, etc.) but has
no idea what documents are stored in Pinecone or what the namespaces are named.

**Solution — probe the `_sections` namespaces:**
- The `_sections` namespace contains one vector per document section (summary-level).
- A metadata query against it gives a high cosine score when the stored document
  is topically similar to the user's study.
- We rank all documents by this probe score and keep only the ones above a threshold.

We probe `_sections` (not `_chunks`) because:
1. Summaries capture the whole-section topic → better document-level relevance signal.
2. Fewer vectors → faster probe.
3. `_chunks` are paragraph fragments that may match a keyword by coincidence.


In [12]:
def score_documents(metadata_query: str, probe_top_k: int = 5, min_score: float = 0.45) -> list:
    """
    Probe every _sections namespace with the metadata query and rank documents.

    Uses the _sections namespace (section summaries) as the relevance signal
    because summaries capture whole-section topics, giving a cleaner document-level
    similarity score than paragraph chunks.

    Args:
        metadata_query: output of build_metadata_query()
        probe_top_k:    top-k hits to pull from each _sections namespace
        min_score:      minimum top-chunk cosine similarity to keep a document

    Returns:
        List of dicts sorted by top_score descending:
        [
          {
            'doc_key':       str,   # e.g. 'Chenodeoxycholic'
            'sections_ns':   str,   # e.g. 'Chenodeoxycholic_sections'
            'chunks_ns':     str,   # e.g. 'Chenodeoxycholic_chunks'
            'top_score':     float, # best single-vector cosine similarity
            'avg_score':     float, # mean across probe_top_k hits
            'selected':      bool,  # True if top_score >= min_score
            'preview':       str,   # snippet of the best matching section
          }
        ]
    """
    vec    = embed(metadata_query)   # embed once, reuse
    scored = []

    for doc_key, (sections_ns, chunks_ns) in NS_PAIRS.items():
        try:
            resp = index.query(
                vector=vec, top_k=probe_top_k,
                include_metadata=True, namespace=sections_ns,
            )
            matches = resp.get('matches', [])
            if not matches:
                continue

            scores   = [m['score'] for m in matches]
            best     = matches[0]
            meta     = best.get('metadata', {})
            preview  = meta.get('text', meta.get('content', ''))[:120].replace('\n', ' ')

            scored.append({
                'doc_key':     doc_key,
                'sections_ns': sections_ns,
                'chunks_ns':   chunks_ns,
                'top_score':   round(max(scores), 4),
                'avg_score':   round(sum(scores) / len(scores), 4),
                'selected':    max(scores) >= min_score,
                'preview':     preview,
            })
        except Exception as e:
            print(f"  [warn] probe failed for {sections_ns}: {e}")

    scored.sort(key=lambda x: x['top_score'], reverse=True)
    return scored


def select_documents(metadata: dict, min_score: float = 0.45, max_docs: int = 2) -> list:
    """
    Derive the most relevant documents from user metadata alone.

    Calls score_documents() with the metadata query, applies the threshold,
    and returns a capped list of the top matching documents.

    Args:
        metadata:  study metadata dict
        min_score: minimum cosine similarity from _sections probe
        max_docs:  maximum number of documents to use (cap to avoid noise)

    Returns:
        List of selected document dicts (same shape as score_documents() output).
        Falls back to the top-1 document if nothing clears the threshold.
    """
    query    = build_metadata_query(metadata)
    scored   = score_documents(query, min_score=min_score)
    selected = [d for d in scored if d['selected']][:max_docs]

    if not selected:
        # Nothing cleared the threshold — use best match as safe fallback
        print("  [warn] No document met the score threshold. Using top-1 as fallback.")
        selected = scored[:1]

    return selected


# ── Run for SAMPLE_METADATA ──────────────────────────────────────────────────
query    = build_metadata_query(SAMPLE_METADATA)
doc_scores = score_documents(query, probe_top_k=5, min_score=0.45)

print(f"Document relevance scores for query:")
print(f"  '{query}...'\n")
print(f"{'Document':<22} {'Top Score':>10}  {'Avg Score':>9}  {'Selected':>8}  Preview")
print('-' * 95)
for d in doc_scores:
    sel = '✅' if d['selected'] else '✗'
    print(f"{d['doc_key']:<22} {d['top_score']:>10}  {d['avg_score']:>9}  {sel:>8}  {d['preview'][:45]}...")

print()
selected_docs = select_documents(SAMPLE_METADATA)
print(f"Selected documents ({len(selected_docs)}): {[d['doc_key'] for d in selected_docs]}")


Document relevance scores for query:
  'Indication: Cerebrotendinous Xanthomatosis (CTX) | Phase: Phase III | Design: Randomized, Double-blind, Placebo-controlled, Multicenter | Primary Endpoint: Reduction in plasma cholestanol levels | Population: Patients aged 2-75 years diagnosed with CTX | Secondary Endpoints: Neurological function improvement (disability scales); MRI-based disease progression; Bone mineral density changes; Safety and tolerability (AE/SAE incidence) | Drug: Chenodeoxycholic Acid (CDCA) | Therapeutic Area: Rare Metabolic Disorders / Neurology...'

Document                Top Score  Avg Score  Selected  Preview
-----------------------------------------------------------------------------------------------
Chenodeoxycholic           0.8387     0.8161         ✅  Section Title: EXPERIMENTAL DESIGN      Purpo...
Epclusa_v1                 0.6574     0.6412         ✅  Section Title: Summary of Additional Clinical...
Amlodipine                 0.6468      0.633         ✅  

## 7. Step 2 — Section-aware retrieval per document section

For each document section we are generating, we perform a **two-pass retrieval**:

**Pass 1 — `_sections` namespace (structural match)**
- Query: `section_keywords | metadata_query`
- Retrieves section-level summaries that match both the section type and the study
- These give the LLM the overall structure and framing of that section in past trials

**Pass 2 — `_chunks` namespace (content match)**
- Same composite query against paragraph-level chunks
- Retrieves the actual detailed content (dosing, criteria, endpoints, stats, etc.)
- If a section fits in one chunk it's stored as-is; if longer it's split into paragraphs

**Merge:** deduplicate by vector ID, keep highest-scoring copy, sort by score, cap at `max_chunks`.

This two-pass approach ensures the LLM gets:
- Section-level framing (from `_sections`) for structure and tone
- Paragraph-level detail (from `_chunks`) for specific values and language


In [13]:
def build_section_query(metadata_query: str, section_id: str) -> str:
    """
    Build a section-specific retrieval query.

    Prepends section keywords to the metadata query so the embedding lands
    in the semantic neighbourhood of that section type.

    Format: "{section_keywords} | {metadata_query}"

    The section keywords come first because sentence-transformers weight
    the beginning of the input slightly more heavily, which helps bias
    retrieval toward section-relevant content.
    """
    keywords = SECTION_MAP[section_id]['keywords']
    return f"{keywords} | {metadata_query}"


def retrieve_for_section(
    metadata_query: str,
    section_id:     str,
    selected_docs:  list,
    top_k_sections: int   = 3,
    top_k_chunks:   int   = 5,
    min_score:      float = 0.30,
    max_total:      int   = 8,
) -> dict:
    """
    Two-pass retrieval for one document section across all selected documents.

    Pass 1: query _sections namespaces  → section summaries (structural context)
    Pass 2: query _chunks namespaces    → paragraph chunks  (detailed content)

    Args:
        metadata_query: output of build_metadata_query()
        section_id:     key from SECTION_MAP
        selected_docs:  output of select_documents() — already filtered by relevance
        top_k_sections: hits per _sections namespace
        top_k_chunks:   hits per _chunks namespace
        min_score:      discard chunks below this cosine similarity
        max_total:      cap on total chunks returned

    Returns:
        {
          'section_id':      str,
          'section_title':   str,
          'section_chunks':  list,   # from _sections — structural context
          'content_chunks':  list,   # from _chunks   — detailed content
          'all_chunks':      list,   # merged, deduplicated, sorted by score
          'sources':         list,   # unique source document names
          'avg_score':       float,
        }
    """
    section_query = build_section_query(metadata_query, section_id)
    vec           = embed(section_query)   # embed once for both passes

    seen      = {}   # id → chunk dict, for deduplication

    def _query_ns(namespace: str, top_k: int, chunk_type: str):
        """Query one namespace and add results to `seen`."""
        try:
            resp = index.query(
                vector=vec, top_k=top_k,
                include_metadata=True, namespace=namespace,
            )
            for m in resp.get('matches', []):
                if m['score'] < min_score:
                    continue
                vid  = m['id']
                meta = m.get('metadata', {})
                chunk = {
                    'id':         vid,
                    'score':      round(m['score'], 4),
                    'text':       meta.get('text', meta.get('content', '')),
                    'metadata':   meta,
                    'namespace':  namespace,
                    'chunk_type': chunk_type,   # 'section_summary' or 'paragraph'
                    'source':     meta.get('filename', meta.get('source', namespace.split('_')[0])),
                }
                # Keep highest-scoring copy when same id appears in multiple namespaces
                if vid not in seen or chunk['score'] > seen[vid]['score']:
                    seen[vid] = chunk
        except Exception as e:
            print(f"    [warn] {namespace}: {e}")

    # ── Pass 1: _sections (structural, summary-level) ────────────────────────
    for doc in selected_docs:
        _query_ns(doc['sections_ns'], top_k_sections, 'section_summary')

    section_ids_after_pass1 = set(seen.keys())

    # ── Pass 2: _chunks (detailed, paragraph-level) ──────────────────────────
    for doc in selected_docs:
        _query_ns(doc['chunks_ns'], top_k_chunks, 'paragraph')

    # ── Merge and sort ────────────────────────────────────────────────────────
    all_chunks     = sorted(seen.values(), key=lambda x: x['score'], reverse=True)[:max_total]
    section_chunks = [c for c in all_chunks if c['id'] in section_ids_after_pass1]
    content_chunks = [c for c in all_chunks if c['id'] not in section_ids_after_pass1]
    sources        = list({c['source'] for c in all_chunks})
    avg_score      = round(sum(c['score'] for c in all_chunks) / max(len(all_chunks), 1), 4)

    return {
        'section_id':     section_id,
        'section_title':  SECTION_MAP[section_id]['title'],
        'section_chunks': section_chunks,
        'content_chunks': content_chunks,
        'all_chunks':     all_chunks,
        'sources':        sources,
        'avg_score':      avg_score,
    }


# ── Quick test: retrieve for one section ─────────────────────────────────────
meta_query    = build_metadata_query(SAMPLE_METADATA)
selected_docs = select_documents(SAMPLE_METADATA)

demo = retrieve_for_section(meta_query, 'study_design', selected_docs)
print(f"Section: {demo['section_title']}")
print(f"  section_summary chunks : {len(demo['section_chunks'])}")
print(f"  paragraph chunks       : {len(demo['content_chunks'])}")
print(f"  total                  : {len(demo['all_chunks'])}")
print(f"  avg_score              : {demo['avg_score']}")
print(f"  sources                : {demo['sources']}")
print()
for c in demo['all_chunks']:
    tag = '[SUM]' if c['chunk_type'] == 'section_summary' else '[PAR]'
    print(f"  {tag} [{c['score']}] ns={c['namespace']}")
    print(f"       {c['text']}\n")
    print()


Section: 2. Study Design
  section_summary chunks : 3
  paragraph chunks       : 5
  total                  : 8
  avg_score              : 0.824
  sources                : ['Chenodeoxycholic']

  [SUM] [0.8589] ns=Chenodeoxycholic_sections
       Section Title: EXPERIMENTAL DESIGN

    Purpose:
To describe the experimental design of a study on cerebrotendinous xanthomatosis.

Key Topics:
- Retrospective study
- Cohort study
- Cerebrotendinous xanthomatosis
- CDCA treatment

Important Entities:
- Medical University of Siena
- Policlinico Le Scotte
- CDCA (chenodeoxycholic acid)

Context Summary:
The study is a retrospective single arm cohort design that collects various types of data from patients with cerebrotendinous xanthomatosis. These patients are treated with CDCA orally at a dosage of 750 mg/day in a specialized neurology unit.


  [PAR] [0.8412] ns=Chenodeoxycholic_chunks
       This is a retrospective single arm cohort study collecting clinical, laboratory and physiological dat

## 8. Step 3 — LLM prompt builder

Formats the retrieved chunks into a structured ICH E6-compliant generation prompt.

The prompt has three parts:
1. **Study details** — the user's metadata (authoritative source of truth)
2. **Retrieved reference material** — section summaries first (structure), then paragraphs (detail)
3. **Instructions** — explicit writing guidelines for the LLM


In [14]:
def format_retrieved_context(retrieval_result: dict, max_chars: int = 500) -> str:
    """
    Format retrieved chunks into a two-part reference block.

    Section summaries come first (structural context),
    followed by paragraph chunks (detailed content).
    Each chunk is numbered and labelled with its type and source.

    Args:
        retrieval_result: output of retrieve_for_section()
        max_chars:        truncate each chunk text to this length

    Returns:
        A formatted string ready to embed in the LLM prompt.
    """
    lines = []
    counter = 1

    # ── Part A: section summaries (structural) ────────────────────────────────
    if retrieval_result['section_chunks']:
        lines.append("--- Section-Level Context (structure and framing) ---")
        for c in retrieval_result['section_chunks']:
            text = c['text'].strip().replace('\n', ' ')
            text = text[:max_chars] + ('...' if len(text) > max_chars else '')
            lines.append(f"[REF {counter}] source={c['source']}  score={c['score']}")
            lines.append(text)
            lines.append("")
            counter += 1

    # ── Part B: paragraph chunks (detailed content) ───────────────────────────
    if retrieval_result['content_chunks']:
        lines.append("--- Detailed Content (specific values and language) ---")
        for c in retrieval_result['content_chunks']:
            text = c['text'].strip().replace('\n', ' ')
            text = text[:max_chars] + ('...' if len(text) > max_chars else '')
            lines.append(f"[REF {counter}] source={c['source']}  score={c['score']}")
            lines.append(text)
            lines.append("")
            counter += 1

    if not lines:
        return "No relevant reference material found in the knowledge base."

    return '\n'.join(lines)


def build_section_prompt(
    metadata:         dict,
    retrieval_result: dict,
    document_type:    str = "Clinical Study Protocol",
) -> str:
    """
    Build the complete LLM generation prompt for one document section.

    This is the prompt that gets passed to groq_client.chat.completions.create()
    (or your existing llm.generate() wrapper) for each section in the loop.

    Args:
        metadata:         study metadata dict
        retrieval_result: output of retrieve_for_section()
        document_type:    e.g. 'Clinical Study Protocol', 'Investigator Brochure'

    Returns:
        Complete prompt string.
    """
    section_title = retrieval_result['section_title']
    rag_context   = format_retrieved_context(retrieval_result)

    indication  = metadata.get('indication', '')
    phase       = metadata.get('phase', '')
    design      = metadata.get('design', '')
    drug        = metadata.get('investigational_product', '[INVESTIGATIONAL_PRODUCT]')
    population  = metadata.get('patient_population', '')
    primary_ep  = metadata.get('primary_endpoint', '')
    sec_eps     = metadata.get('secondary_endpoints', [])
    if isinstance(sec_eps, list):
        sec_eps = '; '.join(sec_eps)
    sample_size = metadata.get('sample_size', 'TBD')
    duration    = metadata.get('duration_months', 'TBD')
    sponsor     = metadata.get('sponsor', '[SPONSOR_NAME]')
    protocol_no = metadata.get('protocol_number', '[PROTOCOL_NUMBER]')
    arms        = metadata.get('treatment_arms', [])
    if isinstance(arms, list):
        arms = '; '.join(arms)

    prompt = f"""You are a senior medical writer drafting a {document_type}.
Write the section titled '{section_title}' for the clinical trial described below.

=== STUDY DETAILS (authoritative — use these exact values) ===
Indication           : {indication}
Phase                : {phase}
Design               : {design}
Investigational Drug : {drug}
Treatment Arms       : {arms}
Patient Population   : {population}
Primary Endpoint     : {primary_ep}
Secondary Endpoints  : {sec_eps}
Sample Size          : {sample_size}
Duration             : {duration} months
Sponsor              : {sponsor}
Protocol Number      : {protocol_no}

=== RETRIEVED REFERENCE MATERIAL ===
The following are excerpts from real past clinical trial documents.
Use them for regulatory language, structure, and precedent — do not copy verbatim.

{rag_context}
=== WRITING INSTRUCTIONS ===
- Write exclusively the content for '{section_title}' — no heading, no preamble.
- Use formal ICH E6(R2) / GCP-compliant clinical trial language.
- Prioritise the study details above as the source of truth for all specific values.
- Use the reference material for tone, structure, and regulatory phrasing.
- Where a required value is not provided, insert a bracketed placeholder: [VALUE].
- Length: 200-400 words appropriate for a {phase} {document_type}.
"""
    return prompt


# ── Preview prompt for 'population' section ──────────────────────────────────
demo_prompt = build_section_prompt(SAMPLE_METADATA, demo)
print(demo_prompt)


You are a senior medical writer drafting a Clinical Study Protocol.
Write the section titled '2. Study Design' for the clinical trial described below.

=== STUDY DETAILS (authoritative — use these exact values) ===
Indication           : Cerebrotendinous Xanthomatosis (CTX)
Phase                : Phase III
Design               : Randomized, Double-blind, Placebo-controlled, Multicenter
Investigational Drug : Chenodeoxycholic Acid (CDCA)
Treatment Arms       : CDCA 750 mg/day orally; Placebo
Patient Population   : Patients aged 2-75 years diagnosed with CTX
Primary Endpoint     : Reduction in plasma cholestanol levels
Secondary Endpoints  : Neurological function improvement (disability scales); MRI-based disease progression; Bone mineral density changes; Safety and tolerability (AE/SAE incidence)
Sample Size          : 120
Duration             : 36 months
Sponsor              : Leadiant Biosciences
Protocol Number      : CTX-CDCA-2024-001

=== RETRIEVED REFERENCE MATERIAL ===
The follow

## 9. Step 4 — Full section-by-section retrieval pipeline

Ties everything together:
1. **Document selection** — probe `_sections` namespaces once, pick relevant documents
2. **Per-section retrieval** — for each section in `SECTION_MAP`, run two-pass retrieval
3. **Prompt assembly** — format each result into an LLM-ready prompt

This mirrors the section generation loop in your backend's `generator_service.py`.


In [15]:
import time

def build_all_section_prompts(
    metadata:       dict,
    document_type:  str   = "Clinical Study Protocol",
    section_ids:    list  = None,
    top_k_sections: int   = 3,
    top_k_chunks:   int   = 5,
    min_score:      float = 0.30,
    max_total:      int   = 8,
    ns_min_score:   float = 0.45,
    delay:          float = 0.2,
) -> tuple:
    """
    Complete retrieval pipeline: document selection → per-section retrieval → prompts.

    The user only provides study metadata. Everything else is automatic.

    Args:
        metadata:       study metadata dict
        document_type:  passed into each section prompt
        section_ids:    sections to generate; None = all SECTION_MAP keys
        top_k_sections: hits per _sections namespace per section
        top_k_chunks:   hits per _chunks namespace per section
        min_score:      minimum cosine similarity for chunks
        max_total:      max chunks per section (merged from both passes)
        ns_min_score:   threshold for document selection probe
        delay:          sleep between sections (avoids Pinecone rate limits)

    Returns:
        (section_results, selected_docs)
        section_results: list of dicts — one per section:
          {
            'section_id':    str,
            'title':         str,
            'prompt':        str,    # ready for LLM
            'retrieval':     dict,   # raw retrieve_for_section() output
            'sources_used':  list,
            'avg_score':     float,
          }
        selected_docs: list of document dicts from select_documents()
    """
    meta_query = build_metadata_query(metadata)

    # ── Step 1: document selection (runs once) ────────────────────────────────
    print("Step 1: Selecting relevant documents...")
    selected = select_documents(metadata, min_score=ns_min_score)
    print(f"  → {len(selected)} document(s): {[d['doc_key'] for d in selected]}\n")

    # ── Step 2: per-section retrieval + prompt building ───────────────────────
    print("Step 2: Retrieving sections...")
    target = section_ids or list(SECTION_MAP.keys())
    results = []

    for sid in target:
        print(f"  [{sid}]", end=' ', flush=True)
        retrieval = retrieve_for_section(
            meta_query, sid, selected,
            top_k_sections=top_k_sections,
            top_k_chunks=top_k_chunks,
            min_score=min_score,
            max_total=max_total,
        )
        prompt = build_section_prompt(metadata, retrieval, document_type)
        print(
            f"{len(retrieval['section_chunks'])} sum + "            f"{len(retrieval['content_chunks'])} par = "            f"{len(retrieval['all_chunks'])} chunks  "            f"avg={retrieval['avg_score']}"            f"  sources={retrieval['sources']}"        )
        results.append({
            'section_id':   sid,
            'title':        retrieval['section_title'],
            'prompt':       prompt,
            'retrieval':    retrieval,
            'sources_used': retrieval['sources'],
            'avg_score':    retrieval['avg_score'],
        })
        if delay:
            time.sleep(delay)

    return results, selected


# ── Run ───────────────────────────────────────────────────────────────────────
print("Running full pipeline for SAMPLE_METADATA...\n")
section_results, used_docs = build_all_section_prompts(SAMPLE_METADATA)

print(f"\n{'='*70}")
print(f"Done: {len(section_results)} sections | documents used: {[d['doc_key'] for d in used_docs]}")
print()
print(f"{'Section':<42} {'Sum':>4} {'Par':>4} {'Tot':>4}  {'Avg':>6}  Sources")
print('-' * 80)
for s in section_results:
    r = s['retrieval']
    print(
        f"{s['title']:<42} "        f"{len(r['section_chunks']):>4} "        f"{len(r['content_chunks']):>4} "        f"{len(r['all_chunks']):>4}  "        f"{s['avg_score']:>6}  "        f"{s['sources_used']}"    )


Running full pipeline for SAMPLE_METADATA...

Step 1: Selecting relevant documents...
  → 2 document(s): ['Chenodeoxycholic', 'Epclusa_v1']

Step 2: Retrieving sections...
  [intro] 3 sum + 5 par = 8 chunks  avg=0.8344  sources=['Chenodeoxycholic']
  [study_design] 3 sum + 5 par = 8 chunks  avg=0.824  sources=['Chenodeoxycholic']
  [objectives] 3 sum + 5 par = 8 chunks  avg=0.8045  sources=['Chenodeoxycholic']
  [population] 3 sum + 5 par = 8 chunks  avg=0.8169  sources=['Chenodeoxycholic']
  [treatment] 3 sum + 5 par = 8 chunks  avg=0.8257  sources=['Chenodeoxycholic']
  [safety] 3 sum + 5 par = 8 chunks  avg=0.8031  sources=['Chenodeoxycholic']
  [statistics] 3 sum + 5 par = 8 chunks  avg=0.8228  sources=['Chenodeoxycholic']
  [ethics] 3 sum + 5 par = 8 chunks  avg=0.8246  sources=['Chenodeoxycholic']
  [data_management] 3 sum + 5 par = 8 chunks  avg=0.8265  sources=['Chenodeoxycholic']

Done: 9 sections | documents used: ['Chenodeoxycholic', 'Epclusa_v1']

Section                   

## 10. Generate sections with Groq

Loop over the section prompts and call the LLM.
Output is a list of `DocumentSection`-shaped dicts matching your backend's `models.py`.


In [ ]:
GROQ_API_KEY = ""       # paste key here or os.getenv('GROQ_API_KEY')
GROQ_MODEL   = "llama-3.3-70b-versatile"
RUN_LLM      = False    # set True when ready

if RUN_LLM and GROQ_API_KEY:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY)
    generated_sections = []

    for sp in section_results:
        print(f"Generating [{sp['section_id']}]...", end=' ', flush=True)
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": sp['prompt']}],
                temperature=0.3,
                max_tokens=700,
            )
            content    = resp.choices[0].message.content.strip()
            word_count = len(content.split())
            print(f"{word_count} words")
        except Exception as e:
            print(f"FAILED: {e}")
            content    = f"[Generation failed: {e}]"
            word_count = 0

        # DocumentSection-shaped dict — matches backend/core/models.py
        generated_sections.append({
            "section_id":       sp['section_id'],
            "title":            sp['title'],
            "content":          content,
            "section_type":     sp['section_id'],
            "word_count":       word_count,
            "confidence_score": sp['avg_score'],
            "sources_used":     sp['sources_used'],
            "compliance_flags": [],
            "revised":          False,
            "revision_count":   0,
            "generated_by":     f"groq/{GROQ_MODEL}",
        })

    # Preview first section
    s0 = generated_sections[0]
    print(f"\nSection: {s0['title']}")
    print(f"Words: {s0['word_count']}  Confidence: {s0['confidence_score']}")
    print(f"Sources: {s0['sources_used']}")
    print("-" * 70)
    print(s0['content'])

else:
    print("LLM call skipped (RUN_LLM=False or no GROQ_API_KEY).")
    print("\nPrompt preview for first section:")
    sp = section_results[0]
    print(f"Section: {sp['title']}")
    print(f"Chunks: {len(sp['retrieval']['all_chunks'])}  avg_score: {sp['avg_score']}")
    print(f"Prompt length: {len(sp['prompt'])} chars")
    print()
    print(sp['prompt'][:1000])


## 11. Drop-in backend integration — `rag_service.py`

The `PineconeRetriever` class below replaces the FAISS retriever in your backend.
Wire it into `generator_service.py` with the four-line pattern at the bottom.


In [ ]:
BACKEND_SNIPPET = '''
# ── backend/services/rag_service.py ─────────────────────────────────────────
# pip install pinecone-client
# .env: PINECONE_API_KEY, PINECONE_INDEX

import time, logging
from pinecone import Pinecone as PineconeClient

logger = logging.getLogger(__name__)

SECTION_KEYWORDS = {
    "intro":           "introduction background rationale disease overview pathophysiology unmet need",
    "study_design":    "study design randomized controlled double-blind treatment arms blinding multicenter",
    "objectives":      "primary endpoint secondary endpoint objectives efficacy outcome measure",
    "population":      "inclusion exclusion criteria patient population eligibility diagnosis",
    "treatment":       "dosage dose regimen administration route treatment duration investigational product",
    "safety":          "adverse events safety monitoring SAE DSMB stopping rules tolerability",
    "statistics":      "statistical analysis sample size power calculation ITT per-protocol",
    "ethics":          "informed consent IRB ethics committee regulatory GCP ICH E6",
    "data_management": "data collection CRF EDC database quality control monitoring",
}


class PineconeRetriever:
    """
    Namespace-aware, section-targeted Pinecone retriever.

    Namespace structure:
        {doc}_sections → section-level summaries  (structural context)
        {doc}_chunks   → paragraph-level chunks    (detailed content)

    Usage in generator_service.py:
        retriever = PineconeRetriever(api_key, index_name, encoder)
        docs      = retriever.select_documents(metadata.to_query_text())
        # then for each section_id in the generation loop:
        chunks    = retriever.retrieve_for_section(
                        metadata.to_query_text(), section_id, docs
                    )
    """

    def __init__(self, api_key: str, index_name: str, encoder,
                 ns_min_score: float = 0.45):
        self._pc           = PineconeClient(api_key=api_key)
        self._index        = self._pc.Index(index_name)
        self._encoder      = encoder
        self._ns_min_score = ns_min_score
        # Build NS_PAIRS from live index stats — zero hardcoding of doc names
        stats = self._index.describe_index_stats()
        all_ns = set(stats.get("namespaces", {}).keys())
        self._ns_pairs = {}
        for ns in all_ns:
            if ns.endswith("_sections"):
                doc_key   = ns[:-len("_sections")]
                chunks_ns = f"{doc_key}_chunks"
                if chunks_ns in all_ns:
                    self._ns_pairs[doc_key] = (ns, chunks_ns)
        logger.info(f"PineconeRetriever: {len(self._ns_pairs)} documents — {list(self._ns_pairs.keys())}")

    def _embed(self, text: str) -> list:
        return self._encoder.encode([text], normalize_embeddings=True)[0].tolist()

    def select_documents(self, metadata_query: str, max_docs: int = 3) -> list:
        """Probe _sections namespaces and return the most relevant documents."""
        vec, scored = self._embed(metadata_query), []
        for doc_key, (sections_ns, chunks_ns) in self._ns_pairs.items():
            try:
                resp = self._index.query(
                    vector=vec, top_k=5, include_metadata=False,
                    namespace=sections_ns
                )
                hits = resp.get("matches", [])
                if hits:
                    top_score = max(m["score"] for m in hits)
                    scored.append((top_score, doc_key, sections_ns, chunks_ns))
            except Exception as e:
                logger.warning(f"Probe failed ns={sections_ns}: {e}")
        scored.sort(reverse=True)
        selected = [
            {"doc_key": dk, "sections_ns": sns, "chunks_ns": cns}
            for score, dk, sns, cns in scored
            if score >= self._ns_min_score
        ][:max_docs]
        if not selected and scored:
            _, dk, sns, cns = scored[0]
            selected = [{"doc_key": dk, "sections_ns": sns, "chunks_ns": cns}]
        logger.info(f"Selected documents: {[d['doc_key'] for d in selected]}")
        return selected

    def retrieve_for_section(
        self, metadata_query: str, section_id: str, selected_docs: list,
        top_k_sections: int = 3, top_k_chunks: int = 5,
        min_score: float = 0.30, max_total: int = 8,
    ) -> list:
        """Two-pass retrieval for one section; returns list of chunk dicts."""
        kw    = SECTION_KEYWORDS.get(section_id, "")
        query = f"{kw} | {metadata_query}" if kw else metadata_query
        vec   = self._embed(query)
        seen  = {}

        def _query(ns, top_k, ctype):
            try:
                resp = self._index.query(vector=vec, top_k=top_k,
                                          include_metadata=True, namespace=ns)
                for m in resp.get("matches", []):
                    if m["score"] >= min_score and m["id"] not in seen:
                        meta = m.get("metadata", {})
                        seen[m["id"]] = {
                            "content":    meta.get("text", meta.get("content", "")),
                            "metadata":   meta,
                            "score":      round(m["score"], 4),
                            "chunk_type": ctype,
                        }
            except Exception as e:
                logger.warning(f"Retrieve failed ns={ns}: {e}")

        for doc in selected_docs:
            _query(doc["sections_ns"], top_k_sections, "section_summary")
        for doc in selected_docs:
            _query(doc["chunks_ns"],   top_k_chunks,   "paragraph")

        return sorted(seen.values(), key=lambda x: x["score"], reverse=True)[:max_total]


# ── Wire into generator_service.py ──────────────────────────────────────────
# In generate(), replace the FAISS retrieve calls with:
#
#   base_query    = metadata.to_query_text()
#   selected_docs = self._pinecone.select_documents(base_query)   # once per request
#
#   for section_id, section_config in SECTION_MAP.items():
#       chunks  = self._pinecone.retrieve_for_section(
#                     base_query, section_id, selected_docs
#                 )
#       prompt  = build_section_prompt(section_id, section_config['title'],
#                                      metadata, chunks, document_type)
#       content = llm.generate(prompt)
'''

print(BACKEND_SNIPPET)


## Summary

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `build_metadata_query()` | Convert study metadata dict to a single query string |
| 2 | `score_documents()` | Probe all `_sections` namespaces, rank documents by cosine similarity |
| 3 | `select_documents()` | Keep only documents above the relevance threshold (auto, no hardcoding) |
| 4 | `build_section_query()` | Prepend section-specific keywords to the metadata query |
| 5 | `retrieve_for_section()` | **Two-pass**: `_sections` (structural) + `_chunks` (detail), merged & deduped |
| 6 | `format_retrieved_context()` | Format chunks into a labelled reference block for the LLM |
| 7 | `build_section_prompt()` | Assemble the full generation prompt (study details + references + instructions) |
| 8 | `build_all_section_prompts()` | Run the full pipeline for all 9 sections in one call |
| 9 | Groq loop | Generate each section, output `DocumentSection`-shaped dicts |
| 10 | `PineconeRetriever` | Drop-in backend class with `select_documents()` + `retrieve_for_section()` |

**Key design decisions:**
- `_sections` namespace is used for **document scoring** (fast, topic-level signal)
- `_sections` + `_chunks` are both used for **content retrieval** (structure + detail)
- Document selection runs **once per user request**, section queries run in a loop
- Namespace pairing is **auto-discovered** from live index stats — zero hardcoded names
- Fallback: if no document clears the threshold, the best-scoring document is used
